In [ ]:
# import pandas as pd

# df1 = pd.read_json("cnndm_gpt35_zero.jsonl", lines=True)
# df1 = df1.rename(columns={"output": "gpt35_summary"})

# df2 = pd.read_json("cnndm_gpt4omini_zero.jsonl", lines=True)
# df2 = df2.rename(columns={"output": "gpt4omini_summary"})

# df3 = pd.read_json("cnndm_llama2_zero.jsonl", lines=True)
# df3 = df3.rename(columns={"output": "llama2_summary"})
# df3.head()

In [ ]:
import pandas as pd
import os

# Define a function to process a JSONL file
def load_and_rename_auto(file_path):
    # Extract the base name of the file (without extension)
    file_name = os.path.basename(file_path)
    
    # Extract the model name (e.g., gpt35, gpt4omini, llama2)
    model_name = file_name.split("_")[1]
    
    # Create the new column name
    new_column_name = f"{model_name}_summary"
    
    # Load the file and rename the 'output' column
    df = pd.read_json(file_path, lines=True)
    return df.rename(columns={"output": new_column_name})

# List of JSONL files to process
file_paths = [
    "cnndm_gpt35_cot.jsonl",
    "cnndm_gpt4omini_cot.jsonl",
    "cnndm_llama2_cot.jsonl",
    "cnndm_llama31_cot.jsonl",
    "cnndm_llama32_cot.jsonl",
    "cnndm_mistral_cot.jsonl",
    "cnndm_orca_cot.jsonl",
    "cnndm_phi35_cot.jsonl",
    "cnndm_vicuna_cot.jsonl",
    
]

dataframes = [load_and_rename_auto(file) for file in file_paths]

# Merge DataFrames on the 'title' column iteratively
merged_df = dataframes[0]
for df in dataframes[1:]:
    merged_df = pd.merge(merged_df, df, on="topic", how="inner")

# Load the CSV file with 'article' and 'highlights' as summary
csv_file_path = "data/generated/cnndm/summarization_llama2_results.csv"
csv_df = pd.read_csv(csv_file_path)

# Rename 'highlights' to 'summary'
csv_df = csv_df.rename(columns={"highlights": "summary"})

# Merge the CSV file with merged_df based on matching 'title' and 'id'
final_df = pd.merge(merged_df, csv_df[['id', 'article', 'summary']], left_on="topic", right_on="id", how="inner")

# Display the final merged DataFrame
final_df

In [ ]:
final_df.to_csv("CNN_cot.csv", index=False)